# تشخیص المان‌های روی برد با YOLO26 (Colab)

همه چیز در **یک پوشه‌ی Drive** است (`runs_hr`): همین نوت‌بوک، داده (`pcb_data_hr.zip`) و آموزش. همه‌ی کدها داخل همین نوت‌بوک است.
لینک پوشه: https://drive.google.com/drive/folders/1yq4gdsJe_sAKt0c10tzWhBHi40pEiiop

**برای هر حساب (فقط یک بار):** لینک بالا را باز کن ▸ روی اسم پوشه راست‌کلیک ▸ `Organize ▸ Add shortcut` ▸ `My Drive`. (حساب صاحب پوشه لازم ندارد.) نوت‌بوک پوشه را با شناسه‌اش پیدا می‌کند، در هر حسابی.

1. `Runtime ▸ Change runtime type ▸ T4 GPU` (اگر GPU نبود روی CPU هم اجرا می‌شود، ولی خیلی کند).
2. `Runtime ▸ Run all`.
3. اگر Colab قطع شد: دوباره Run all. هر بلوکی که کارش تمام شده خودش را می‌پرد و آموزش از آخرین epoch ادامه می‌دهد.
4. **هم‌زمان فقط یک نفر آموزش را اجرا کند** (همه در یک پوشه ذخیره می‌کنند).

| بلوک | کار |
|---|---|
| ⚙️ تنظیمات | تنها جایی که لازم است عوض کنی |
| 1 نصب | نصب و به‌روزرسانی کتابخانه‌ها، اتصال به Drive، دیدن GPU |
| 2 کد | سه فایل کد (بررسی داده، آموزش، تست مدل) |
| 3 داده | باز کردن `pcb_data_hr.zip` |
| 4 بررسی داده | اگر مشکلی باشد قبل از آموزش متوقف می‌شود |
| 5 وضعیت | آموزش تا کجا رفته (بدون اجرای آموزش) |
| 6 آموزش | مرحله‌ی `any` (هر المانی را پیدا کن)، سپس `fine` (نامش را بگو) - با خروجی و نمودار زنده |
| 7 نتیجه | دقت روی بردهای تست و نمونه‌ی خروجی |
| 8 تست روی عکس خودت | هر عکس برد را بده و خروجی را ببین |

In [ ]:
# ⚙️ تنظیمات - فقط همین بلوک را در صورت نیاز تغییر بده
RUNS_DIR    = ""                             # خالی = پوشه‌ی آموزش با شناسه‌ی Drive خودکار پیدا می‌شود (پیشنهاد)
RUNS_FOLDER_ID = "1yq4gdsJe_sAKt0c10tzWhBHi40pEiiop"   # شناسه‌ی پوشه‌ی آموزش در Drive (همه‌ی حساب‌ها همین یک پوشه را می‌بینند)
RUNS_NAME   = "runs_hr"                       # اسم همان پوشه (وزن‌ها، نمودارها، نتیجه‌ها، داده و نوت‌بوک)
MODEL       = "yolo26s.pt"                   # yolo26n.pt (سبک) | yolo26s.pt | yolo26m.pt (دقیق‌تر، کندتر)
IMGSZ       = 1024                           # اگر حافظه‌ی GPU کم آمد 800 بگذار
EPOCHS      = 80                           # هر epoch در دیتاست تکه‌تکه داده‌ی بیشتری دارد؛ اگر 30 epoch بهتر نشود خودش می‌ایستد
STAGES      = "any fine"                     # "any" = فقط پیدا کردن، "fine" = نام‌گذاری، یا هر دو
SESSION_HOURS = 0                            # مثلا 3.5: قبل از این همه ساعت، بعد از ذخیره‌ی یک epoch، آموزش تمیز متوقف می‌شود (0 = خاموش)
BATCH       = "8"                            # اگر خطای حافظه (out of memory) آمد "4" بگذار؛ "-1" = خودکار
EXTRA       = "--cache disk"                             # آرگومان اضافه برای آموزش، مثلا "--patience 50" یا "--cos-lr" یا "--export onnx"
SKIP_INSTALL = False                         # True اگر مطمئنی نصب است
ZIP_FILE_ID = "1RZFp4TYgWJdIV3RBGg7Pw7tt9ilmM3CB"   # اگر zip در Drive نبود از این لینک دانلود می‌شود (gdown)
ALLOW_NEW_RUN = False      # False: اگر آموزش قبلی (last.pt) در RUNS پیدا نشود، آموزش از صفر شروع نمی‌شود

## 1) نصب و به‌روزرسانی

In [ ]:
import os, sys, subprocess
WORK = "/content/work"   # دیسک سریع Colab (با قطع شدن پاک می‌شود؛ نتایج روی Drive می‌مانند)

def run(cmd, check=True, watch=None):
    """Run a command and show its output LIVE in this cell (plain subprocess.run shows nothing in Colab).
    watch: glob of an image (e.g. progress.png) that is shown and refreshed after every epoch."""
    import codecs, glob as _glob
    print("$", " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    dec = codecs.getincrementaldecoder("utf8")("replace")
    shown = {}
    def refresh():
        try:
            from IPython.display import Image, display
        except ImportError:
            return
        for f in sorted(_glob.glob(watch)):
            if f in shown:
                shown[f].update(Image(f, width=1100))
            else:
                shown[f] = display(Image(f, width=1100), display_id=True)
    try:
        for chunk in iter(lambda: p.stdout.read1(4096), b""):
            text = dec.decode(chunk)
            sys.stdout.write(text); sys.stdout.flush()
            if watch and "| saved" in text:
                refresh()
    except KeyboardInterrupt:
        p.terminate(); p.wait(); print("\nstopped (run the block again to resume)"); raise
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed (exit code {p.returncode}) - read the output above")
    return p.returncode

def sh(*cmd):
    run(cmd)

# اتصال به Drive (خارج از Colab بی‌خطر رد می‌شود)
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Colab نیست - از پوشه‌ی محلی استفاده می‌شود")

def drive_id(path):
    try:
        return os.getxattr(path, "user.drive.id").decode()
    except Exception:
        return None

def find_runs():
    """The training folder is the same Drive folder for every account: the owner sees it in My Drive,
    others through a shortcut. It is recognised by its Drive ID, so the right folder is used everywhere."""
    import glob as _g
    if RUNS_DIR:
        return RUNS_DIR
    pats = [f"/content/drive/.shortcut-targets-by-id/{RUNS_FOLDER_ID}/{RUNS_NAME}",
            f"/content/drive/MyDrive/{RUNS_NAME}", f"/content/drive/MyDrive/*/{RUNS_NAME}", f"/content/drive/MyDrive/*/*/{RUNS_NAME}",
            f"/content/drive/.shortcut-targets-by-id/*/{RUNS_NAME}", f"/content/drive/.shortcut-targets-by-id/*/*/{RUNS_NAME}"]
    cands = [c for pat in pats for c in _g.glob(pat) if os.path.isdir(c)]
    for c in cands:
        if RUNS_FOLDER_ID in c or drive_id(c) == RUNS_FOLDER_ID:
            return c
    with_ck = [c for c in cands if _g.glob(f"{c}/*/weights/last.pt")]
    if with_ck:   # Drive IDs not readable on this runtime: fall back to the folder that holds the training
        print("⚠️ شناسه‌ی Drive خوانده نشد؛ پوشه‌ای که آموزش در آن است انتخاب شد:", with_ck)
        return max(with_ck, key=lambda c: len(_g.glob(f"{c}/*/weights/*.pt")))
    raise FileNotFoundError(
        f"پوشه‌ی آموزش ({RUNS_NAME}) در این حساب دیده نمی‌شود. این لینک را باز کن: https://drive.google.com/drive/folders/{RUNS_FOLDER_ID} "
        "▸ راست‌کلیک روی اسم پوشه ▸ Organize ▸ Add shortcut ▸ My Drive. بعد Runtime ▸ Disconnect and delete runtime و دوباره Run all.")

RUNS = find_runs()
DRIVE_DIR = RUNS            # data zip and notebook live in the same folder
print("پوشه‌ی آموزش و داده:", RUNS)

if not SKIP_INSTALL:
    # نسخه‌ی ثابت: همان نسخه‌ای که کد آموزش با آن آزمایش شده؛ برای همه یکسان است
    sh(sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.174", "pyyaml")

import ultralytics, torch, cv2, yaml
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| opencv", cv2.__version__)
ver = tuple(int(x) for x in ultralytics.__version__.split(".")[:2])
assert ver >= (8, 4), "ultralytics قدیمی است (برای YOLO26 نسخه‌ی 8.4 یا بالاتر لازم است): این بلوک را دوباره اجرا کن یا Runtime را Restart کن"
GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if GPU else "ندارد -> آموزش روی CPU انجام می‌شود (خیلی کندتر؛ اگر شد Runtime را روی T4 GPU بگذار)")
os.makedirs(WORK, exist_ok=True); os.makedirs(RUNS, exist_ok=True)

## 2) کد (سه فایل؛ هر کدام را می‌شود همین‌جا ویرایش کرد)

In [ ]:
%%writefile /content/work/verify_dataset.py
"""Check a built dataset before training (runs locally and on Colab, takes about a minute).

Checks: every image has a label file; label values in range; class ids valid; images are in colour;
no board appears in two splits (needs split_manifest.csv); boxes per split/class; tiny-box share.
Exit code 1 when a hard error is found, so a pipeline can stop before wasting GPU time.

  python tools/verify_dataset.py --root final_dataset
"""
import argparse
import csv
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np
import yaml


def check_view(root, view):
    d = root / view
    cfg = yaml.safe_load(open(d / "data.yaml", encoding="utf8"))
    names = list(cfg["names"].values()) if isinstance(cfg["names"], dict) else cfg["names"]
    out = {"errors": [], "warnings": [], "splits": {}}
    img_root = d / "images" if (d / "images").exists() else root / "fine" / "images"   # HR zip stores pictures once
    for sp in ("train", "val", "test"):
        imgs = sorted((img_root / sp).glob("*.jpg"))
        cnt, tiny, n_box, sat_low, empty = Counter(), 0, 0, 0, 0
        for im in imgs:
            lbl = d / "labels" / sp / (im.stem + ".txt")
            if not lbl.exists():
                out["errors"].append(f"{view}/{sp}: missing label for {im.name}")
                continue
            lines = [l.split() for l in lbl.read_text(encoding="utf8").splitlines() if l.strip()]
            if not lines:
                empty += 1
            for p in lines:
                try:
                    c, x, y, w, h = int(p[0]), *map(float, p[1:5])
                except Exception:
                    out["errors"].append(f"{view}/{sp}: bad line in {lbl.name}: {p}")
                    continue
                if not (0 <= c < len(names)) or not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                    out["errors"].append(f"{view}/{sp}: out-of-range value in {lbl.name}: {p}")
                    continue
                cnt[names[c]] += 1; n_box += 1
                tiny += min(w, h) * 640 < 4
            if len(imgs) <= 120 or im.name.endswith("_r90.jpg") is False:
                img = cv2.imread(str(im))
                if float(cv2.cvtColor(img, cv2.COLOR_BGR2HSV)[..., 1].mean()) < 3:
                    sat_low += 1
        out["splits"][sp] = {"images": len(imgs), "boxes": n_box, "empty_label_files": empty,
                             "boxes_under_4px_at_640": tiny, "grayscale_images": sat_low, "per_class": dict(cnt)}
        if sat_low:
            out["errors"].append(f"{view}/{sp}: {sat_low} grayscale images")
        if not imgs:
            out["errors"].append(f"{view}/{sp}: no images")
        for n in names:
            if sp != "train" and cnt[n] < 10:
                out["warnings"].append(f"{view}/{sp}: class '{n}' has only {cnt[n]} boxes (its metric will be noisy)")
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", default="final_dataset")
    a = ap.parse_args()
    root = Path(a.root)
    report = {v: check_view(root, v) for v in ("any", "fine")}

    man = root / "split_manifest.csv"
    if man.exists():
        boards = defaultdict(set)
        for r in csv.DictReader(open(man, encoding="utf8")):
            boards[r["board"]].add(r["split"])
        leaks = [b for b, s in boards.items() if len(s) > 1]
        report["boards"] = len(boards)
        report["boards_in_several_splits"] = len(leaks)
        if leaks:
            report["fine"]["errors"].append(f"{len(leaks)} boards appear in more than one split")

    errors = report["any"]["errors"] + report["fine"]["errors"]
    (root / "qa_report.json").write_text(json.dumps(report, indent=1), encoding="utf8")
    for v in ("any", "fine"):
        print(f"[{v}]", {sp: (s["images"], s["boxes"]) for sp, s in report[v]["splits"].items()}, "(images, boxes)")
    print("boards:", report.get("boards"), " boards in several splits:", report.get("boards_in_several_splits"))
    tiny = sum(s["boxes_under_4px_at_640"] for s in report["fine"]["splits"].values())
    tot = sum(s["boxes"] for s in report["fine"]["splits"].values())
    print(f"boxes thinner than 4 px at 640: {tiny}/{tot} ({100 * tiny / max(tot, 1):.0f}%)")
    for w in report["fine"]["warnings"][:10]:
        print("WARN", w)
    if errors:
        print(f"\n{len(errors)} ERRORS, first ones:")
        for e in errors[:10]:
            print("  ", e)
        sys.exit(1)
    print("DATASET OK")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/work/train_yolo26.py
"""Train YOLO26 in two stages with checkpoints that survive a Colab disconnect.

  stage "any"  : one class "component"  -> find every element on the board, whatever it is
  stage "fine" : resistor / capacitor / IC ... -> optionally starts from the "any" weights

Everything (last.pt every epoch, best.pt, snapshots, metrics, progress) is written to --project,
which should be a Google Drive folder.  Run the same command again after a disconnect
and every stage resumes by itself; finished stages are skipped.

Per stage, in <project>/<stage>_<model>/ :
  progress.txt   one line per epoch (losses, val metrics, best epoch, speed, ETA)
  progress.png   live training curves, redrawn every epoch
  results.csv    Ultralytics' own per-epoch table
  weights/       last.pt (every epoch), last_prev.pt (one epoch older, used if last.pt is damaged), best.pt
  test_metrics.json  final test-set metrics, per class too

  python train_yolo26.py --data-root /content/final_dataset --project /content/drive/MyDrive/pcb_runs
"""
import argparse
import csv
import json
import os
import shutil
import sys
import time
from pathlib import Path

import torch
import yaml
from ultralytics import YOLO

T0 = time.time()

# Augmentation: colour is kept (no grayscale), geometry is mild so boxes stay tight.
AUG = dict(
    hsv_h=0.01, hsv_s=0.4, hsv_v=0.3,        # lighting / camera differences; colour is kept (there is no grayscale augmentation)
    degrees=5.0, translate=0.10, scale=0.5, shear=0.0, perspective=0.0,
    flipud=0.5, fliplr=0.5,                  # a board can be seen from any side
    mosaic=1.0, close_mosaic=10, mixup=0.0, copy_paste=0.0,
)


class SessionTimeUp(Exception):
    """Raised after a checkpoint is saved when --hours is used up; the run resumes next time."""


def hms(s):
    s = int(max(s, 0))
    return f"{s // 3600}h{s % 3600 // 60:02d}m" if s >= 3600 else f"{s // 60}m{s % 60:02d}s"


def safe_copy(src, dst):
    tmp = Path(str(dst) + ".tmp")
    shutil.copyfile(src, tmp)
    os.replace(tmp, dst)


def loads(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except Exception as e:
        print(f"   ! {path.name} cannot be read ({type(e).__name__}: {e})", flush=True)
        return None


def finished(ckpt):
    """Ultralytics marks a checkpoint as finished (epoch = -1) when training ends; such a file cannot be resumed."""
    return ckpt.get("epoch", -1) == -1


def plot_progress(run_dir, stage):
    """Redraw progress.png from results.csv (cheap, a few hundred ms)."""
    f = run_dir / "results.csv"
    if not f.exists():
        return
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    rows = [{k.strip(): v for k, v in r.items()} for r in csv.DictReader(open(f, encoding="utf8"))]
    rows = sorted({int(float(r["epoch"])): r for r in rows}.values(), key=lambda r: float(r["epoch"]))  # repeated epochs after a fallback: keep the latest
    if not rows:
        return
    col = lambda k: [float(r[k]) for r in rows if r.get(k, "").strip()]
    ep = col("epoch")
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
    losses = [k[6:] for k in rows[0] if k.startswith("train/") and k.endswith("_loss")]  # box/cls/dfl or box/cls/l1 ...
    for i, k in enumerate(losses):
        ax[0].plot(ep, col(f"train/{k}"), f"C{i}", label=f"train {k[:-5]}")
        if len(col(f"val/{k}")) == len(ep):
            ax[0].plot(ep, col(f"val/{k}"), f"C{i}--", label=f"val {k[:-5]}")
    ax[0].set_title("loss (lower is better)")
    for k, lab in (("metrics/mAP50(B)", "mAP50"), ("metrics/mAP50-95(B)", "mAP50-95")):
        ax[1].plot(ep, col(k), marker=".", label=lab)
    m = col("metrics/mAP50-95(B)")
    if m:
        b = max(range(len(m)), key=m.__getitem__)
        ax[1].axvline(ep[b], color="grey", ls=":", label=f"best epoch {int(ep[b])}")
    ax[1].set_title("val accuracy (higher is better)")
    ax[2].plot(ep, col("metrics/precision(B)"), label="precision")
    ax[2].plot(ep, col("metrics/recall(B)"), label="recall")
    ax[2].set_title("precision / recall")
    for x in ax:
        x.grid(alpha=.3); x.legend(fontsize=7); x.set_xlabel("epoch")
    fig.suptitle(f"stage '{stage}' - {len(ep)} epochs - updated {time.strftime('%H:%M')}")
    fig.tight_layout()
    tmp = run_dir / "progress.tmp.png"
    fig.savefig(tmp, dpi=80); plt.close(fig)
    os.replace(tmp, run_dir / "progress.png")


def add_callbacks(model, stage, run_dir, a):
    """One clear line per epoch (also in progress.txt), live curves, a backup of the previous checkpoint,
    and a clean pause when the session budget (--hours) is used up."""
    log = run_dir / "progress.txt"
    last, prev = run_dir / "weights" / "last.pt", run_dir / "weights" / "last_prev.pt"
    seen = set()

    def say(msg):
        print(msg, flush=True)
        with open(log, "a", encoding="utf8") as fh:
            fh.write(msg + "\n")

    def on_train_start(tr):
        n_train = len(tr.train_loader.dataset) if getattr(tr, "train_loader", None) else "?"
        if torch.cuda.is_available():
            p = torch.cuda.get_device_properties(0)
            gpu = f"{p.name} {p.total_memory / 2**30:.0f} GB"
        else:
            gpu = "CPU"
        say(f"\n{'=' * 78}\n[{stage}] TRAINING START  {time.strftime('%Y-%m-%d %H:%M')}  | {gpu}\n"
            f"[{stage}] epochs {tr.start_epoch + 1}->{tr.epochs} | train images {n_train} | batch {tr.batch_size} "
            f"| imgsz {tr.args.imgsz} | classes {tr.data['nc']} | patience {tr.args.patience}\n{'=' * 78}")

    def on_train_epoch_end(tr):
        # runs right before Ultralytics overwrites last.pt: keep the previous one as a fallback
        if last.exists():
            try:
                safe_copy(last, prev)
            except OSError as e:
                print(f"   ! backup of last.pt failed: {e}", flush=True)

    def on_fit_epoch_end(tr):
        # the progress line is only a report: a problem in it (e.g. a newer Ultralytics) must never stop training
        try:
            report(tr)
        except SessionTimeUp:
            raise
        except Exception as e:
            print(f"   ! progress line failed ({type(e).__name__}: {e}) - training continues", flush=True)

    def loss_text(tr):
        t = tr.tloss  # tensor in older Ultralytics, {name: value} dict in newer ones
        if t is None:
            return ""
        items = t.items() if isinstance(t, dict) else zip(tr.loss_names, t)
        return "  ".join(f"{str(n).replace('_loss', '')} {float(v):.3f}" for n, v in items)

    def report(tr):
        m = tr.metrics or {}
        g = lambda k: float(m.get(k, 0.0))
        val = (f"P {g('metrics/precision(B)'):.3f}  R {g('metrics/recall(B)'):.3f}  "
               f"mAP50 {g('metrics/mAP50(B)'):.3f}  mAP50-95 {g('metrics/mAP50-95(B)'):.3f}")
        if tr.epoch in seen or tr.epoch >= tr.epochs:  # Ultralytics calls this once more after the final validation of best.pt
            say(f"[{stage}] FINAL (best.pt on val) | {val}")
            return
        seen.add(tr.epoch)
        ep = tr.epoch + 1
        loss = loss_text(tr)
        st, dt = tr.stopper, tr.epoch_time or 0
        mem = f" | GPU {torch.cuda.max_memory_reserved() / 2**30:.1f}G" if torch.cuda.is_available() else ""
        say(f"[{stage}] epoch {ep:>3}/{tr.epochs} | loss {loss} | val {val} | best epoch {st.best_epoch} "
            f"(no gain {ep - st.best_epoch}/{tr.args.patience}) | {hms(dt)}/epoch, ETA {hms(dt * (tr.epochs - ep))}"
            f"{mem} | session {hms(time.time() - T0)} | saved")
        try:
            plot_progress(run_dir, stage)
        except Exception as e:  # a plot must never stop training
            print(f"   ! progress plot failed: {e}", flush=True)
        if a.hours and ep < tr.epochs and time.time() - T0 + 1.2 * dt > a.hours * 3600:
            raise SessionTimeUp

    model.add_callback("on_train_start", on_train_start)
    model.add_callback("on_train_epoch_end", on_train_epoch_end)
    model.add_callback("on_fit_epoch_end", on_fit_epoch_end)


def pick_checkpoint(run_dir):
    """Return ('resume' | 'finished' | 'new', ckpt path). If last.pt is damaged, falls back to last_prev.pt
    (one epoch older), then to the newest epochN.pt snapshot."""
    w = run_dir / "weights"
    last, prev = w / "last.pt", w / "last_prev.pt"
    snaps = sorted(w.glob("epoch*.pt"), key=lambda p: -int(p.stem[5:]) if p.stem[5:].isdigit() else 0)
    tried = False
    for p in (last, prev, *snaps):
        if not p.exists():
            continue
        tried = True
        ck = loads(p)
        if ck is None:
            continue
        if p != last:
            print(f"   using the backup {p.name} instead of the damaged last.pt", flush=True)
            if last.exists():
                os.replace(last, w / "last_damaged.pt")
            safe_copy(p, last)
        return ("finished" if finished(ck) else "resume"), last
    if tried:
        print("   ! no readable checkpoint found - this stage starts again from the beginning", flush=True)
    return "new", None


def retarget(ckpt_path, run_dir, data):
    """Ultralytics resumes into the absolute folder stored inside last.pt. When the run was started on another
    Google account / Drive folder that path is wrong here, so training would write somewhere else (or look like
    it starts over). Point the checkpoint at the folder it actually lives in now."""
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    ta = ck.get("train_args") or {}
    want = {"project": str(run_dir.parent), "name": run_dir.name, "save_dir": str(run_dir), "data": data, "exist_ok": True}
    old = {k: ta.get(k) for k in want}
    if any(str(old[k]) != str(v) for k, v in want.items()):
        ta.update(want)
        ck["train_args"] = ta
        tmp = Path(str(ckpt_path) + ".tmp")
        torch.save(ck, tmp)
        os.replace(tmp, ckpt_path)
        print(f"   checkpoint moved: {old['save_dir'] or old['project']}  ->  {run_dir}", flush=True)


def train_kwargs(a):
    """Settings the user gave on the command line that may be changed when resuming (e.g. a smaller batch on another GPU)."""
    out = {k: getattr(a, k) for k in ("batch", "workers", "cache", "device", "patience") if getattr(a, k) is not None}
    return out


def run_stage(stage, a):
    run_dir = Path(a.project) / f"{stage}_{Path(a.model).stem}"
    best, done = run_dir / "weights" / "best.pt", run_dir / "DONE"
    # data.yaml was written on another computer: point its "path" at the real folder
    ds = Path(a.data_root).resolve() / stage
    cfg = yaml.safe_load(open(ds / "data.yaml", encoding="utf8"))
    cfg["path"] = str(ds)
    run_dir.mkdir(parents=True, exist_ok=True)
    data = str(run_dir / "data_local.yaml")
    yaml.safe_dump(cfg, open(data, "w", encoding="utf8"), allow_unicode=True, sort_keys=False)

    if done.exists():
        print(f"[{stage}] already finished -> {best}", flush=True)
        return best
    state, last = pick_checkpoint(run_dir)
    if state == "finished" and best.exists():
        # training ended but the test step below did not (e.g. disconnect): just redo the test step
        print(f"[{stage}] training already complete, running the test evaluation only", flush=True)
    elif state == "resume":
        print(f"[{stage}] resuming from {last}", flush=True)
        retarget(last, run_dir, data)
        model = YOLO(str(last))
        add_callbacks(model, stage, run_dir, a)
        model.train(resume=True, **train_kwargs(a))
    else:
        start = a.model
        init = Path(a.project) / f"any_{Path(a.model).stem}" / "weights" / "best.pt"
        if stage == "fine" and not a.scratch and init.exists():
            start = str(init)
            print(f"[fine] initialising from the 'any' detector: {init}", flush=True)
        model = YOLO(start)
        add_callbacks(model, stage, run_dir, a)
        # batch 8: the automatic batch size picked 1 on Colab's T4 (slow, noisy); boards have hundreds of parts
        # per image, so max_det is raised from 300 for the validation during training
        kw = dict(batch=8, workers=4, cache="ram", patience=30) | train_kwargs(a)
        model.train(
            data=data, epochs=a.epochs, imgsz=a.imgsz, project=a.project, name=run_dir.name, exist_ok=True,
            save=True, save_period=a.save_period, seed=0, plots=True, fraction=a.fraction, cos_lr=a.cos_lr,
            max_det=a.max_det, **kw, **AUG,
        )
    plot_progress(run_dir, stage)

    # always keep a copy of the best weights and the test metrics next to the run
    print(f"[{stage}] evaluating best.pt on the TEST boards ...", flush=True)
    best_model = YOLO(str(best))
    m = best_model.val(data=data, split="test", imgsz=a.imgsz, plots=True, device=a.device, max_det=a.max_det,
                       project=str(run_dir), name="test", exist_ok=True)
    metrics = {"precision": float(m.box.mp), "recall": float(m.box.mr),
               "mAP50": float(m.box.map50), "mAP50-95": float(m.box.map), "per_class": {}}
    for i, c in enumerate(m.ap_class_index):
        p, r, ap50, ap = m.box.class_result(i)
        metrics["per_class"][best_model.names[int(c)]] = {"precision": round(float(p), 4), "recall": round(float(r), 4),
                                                         "mAP50": round(float(ap50), 4), "mAP50-95": round(float(ap), 4)}
    (run_dir / "test_metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf8")
    final = Path(a.project) / "final"
    final.mkdir(exist_ok=True)
    shutil.copy2(best, final / f"{stage}_best.pt")
    if a.export:
        f = best_model.export(format=a.export, imgsz=a.imgsz)
        shutil.copy2(f, final / f"{stage}_best{Path(f).suffix}")
    done.write_text("ok")

    print(f"\n[{stage}] TEST  P {metrics['precision']:.3f}  R {metrics['recall']:.3f}  "
          f"mAP50 {metrics['mAP50']:.3f}  mAP50-95 {metrics['mAP50-95']:.3f}", flush=True)
    if len(metrics["per_class"]) > 1:
        print(f"   {'class':<24}{'P':>7}{'R':>7}{'mAP50':>8}{'mAP50-95':>10}")
        for n, v in sorted(metrics["per_class"].items(), key=lambda kv: -kv[1]["mAP50"]):
            print(f"   {n:<24}{v['precision']:>7.3f}{v['recall']:>7.3f}{v['mAP50']:>8.3f}{v['mAP50-95']:>10.3f}")
    print(f"[{stage}] best weights -> {final / (stage + '_best.pt')}", flush=True)
    return best


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-root", required=True, help="folder that contains any/ and fine/ (final_dataset)")
    ap.add_argument("--project", required=True, help="output folder, put it on Google Drive")
    ap.add_argument("--stages", nargs="+", default=["any", "fine"], choices=["any", "fine"])
    ap.add_argument("--model", default="yolo26s.pt")
    ap.add_argument("--imgsz", type=int, default=1024)
    ap.add_argument("--epochs", type=int, default=100)
    ap.add_argument("--batch", type=float, default=None, help="8 by default, -1 = auto; also allowed when resuming, e.g. on a smaller GPU")
    ap.add_argument("--max-det", type=int, default=1000, help="max detections per image in validation/test (boards have many parts)")
    ap.add_argument("--patience", type=int, default=None, help="stop after this many epochs without improvement (default 30)")
    ap.add_argument("--save-period", type=int, default=10, help="extra snapshot every N epochs (last.pt is every epoch)")
    ap.add_argument("--cache", default=None, help="ram (default) | disk | False")
    ap.add_argument("--workers", type=int, default=None, help="dataloader workers (default 4)")
    ap.add_argument("--fraction", type=float, default=1.0, help="use only this share of the training set (dry runs)")
    ap.add_argument("--device", default=None, help="e.g. 0 or cpu (default: auto)")
    ap.add_argument("--cos-lr", action="store_true", help="cosine learning-rate schedule (new runs only)")
    ap.add_argument("--scratch", action="store_true", help="do not start 'fine' from the 'any' weights")
    ap.add_argument("--hours", type=float, default=0, help="pause cleanly (after a saved epoch) before this many hours; 0 = off")
    ap.add_argument("--export", default="", help="also export the best model, e.g. onnx")
    ap.add_argument("--allow-cpu", action="store_true", help="train even without a GPU (very slow)")
    args = ap.parse_args()
    if args.batch is not None:
        args.batch = int(args.batch) if args.batch == int(args.batch) else args.batch
    if args.cache == "False":
        args.cache = False
    if not torch.cuda.is_available() and str(args.device) != "cpu" and not args.allow_cpu:
        sys.exit("No GPU found. In Colab: Runtime > Change runtime type > GPU (or pass --allow-cpu).")
    try:
        for s in args.stages:
            run_stage(s, args)
    except SessionTimeUp:
        print(f"\nSession budget of {args.hours} h reached - stopped after a saved epoch. "
              f"Run the same block again (in a new session) to continue.", flush=True)

In [ ]:
%%writefile /content/work/predict.py
"""Quick test of trained models on board photos, with a clean (non-cluttered) drawing.

Modes
  any   : the "component" detector only        -> every element, type unknown
  fine  : the multi-class detector only
  both  : "any" finds the elements, "fine" names them; elements the fine model cannot
          name get the label "?" (these are the ones to hand to the random forest)

Examples
  python predict.py --any final/any_best.pt --source board.jpg --mode any
  python predict.py --any final/any_best.pt --fine final/fine_best.pt --source boards/ --mode both --tile 640
  python predict.py --fine final/fine_best.pt --source board.jpg --mode fine --conf 0.15 --labels names

Outputs (in --out): <name>_elements.jpg (where: every part in one colour), <name>_types.jpg (what: colour per type),
                    <name>_detections.csv, <name>_summary.json
"""
import argparse
import colorsys
import csv
import json
from pathlib import Path

import cv2
import numpy as np
import torch
from torchvision.ops import batched_nms, box_iou
from ultralytics import YOLO

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


# --------------------------------------------------------------- detection
def detect(model, img, conf, imgsz, tile, overlap, iou, device, max_det=1000):
    """Return tensors boxes[N,4] (xyxy), scores[N], cls[N]; tiled when tile > 0."""
    H, W = img.shape[:2]
    if not tile or max(H, W) <= tile:
        r = model.predict(img, conf=conf, imgsz=imgsz, device=device, max_det=max_det, verbose=False)[0]
        return r.boxes.xyxy.cpu(), r.boxes.conf.cpu(), r.boxes.cls.cpu().long()
    step = max(1, int(tile * (1 - overlap)))
    xs = list(range(0, max(1, W - tile) + 1, step)); ys = list(range(0, max(1, H - tile) + 1, step))
    if xs[-1] + tile < W: xs.append(W - tile)
    if ys[-1] + tile < H: ys.append(H - tile)
    B, S, C = [], [], []
    for y in ys:
        for x in xs:
            r = model.predict(img[y:y + tile, x:x + tile], conf=conf, imgsz=imgsz, device=device, max_det=max_det, verbose=False)[0]
            b = r.boxes.xyxy.cpu().clone(); b[:, [0, 2]] += x; b[:, [1, 3]] += y
            B.append(b); S.append(r.boxes.conf.cpu()); C.append(r.boxes.cls.cpu().long())
    B, S, C = torch.cat(B), torch.cat(S), torch.cat(C)
    keep = batched_nms(B, S, C, iou)
    return B[keep], S[keep], C[keep]


def agnostic_nms(B, S, C, iou):
    keep = batched_nms(B, S, torch.zeros_like(C), iou)
    return B[keep], S[keep], C[keep]


def combine(any_det, fine_det, names, match_iou=0.3):
    """Name every 'any' box with the best-overlapping fine box; otherwise '?' (-> random forest)."""
    Ba, Sa, _ = any_det
    Bf, Sf, Cf = fine_det
    labels, conf_cls = [], []
    iou_m = box_iou(Ba, Bf) if len(Ba) and len(Bf) else torch.zeros(len(Ba), len(Bf))
    for i in range(len(Ba)):
        if iou_m.shape[1] and iou_m[i].max() >= match_iou:
            j = int(iou_m[i].argmax()); labels.append(names[int(Cf[j])]); conf_cls.append(float(Sf[j]))
        else:
            labels.append("?"); conf_cls.append(0.0)
    return labels, conf_cls


# --------------------------------------------------------------- drawing
SHORT = {"Resistor": "R", "Capacitor": "C", "Electrolytic Capacitor": "EC", "IC": "IC", "Inductor": "L", "Diode": "D",
         "LED": "LED", "Transistor": "Q", "Oscillator": "X", "Connector": "CN", "Switch/Button": "SW", "Other": "O",
         "Marker": "M", "component": "", "?": "?"}


def palette(labels):
    uniq = sorted(set(labels), key=lambda s: (s == "?", s))
    out = {}
    for i, k in enumerate(uniq):
        r, g, b = colorsys.hsv_to_rgb((i * 0.61803) % 1.0, 0.9, 1.0)
        out[k] = (int(b * 255), int(g * 255), int(r * 255))
    if "?" in out:
        out["?"] = (210, 210, 210)
    return out


ELEMENT_COLOUR = (0, 255, 120)   # one bright green for "where are the parts"


def _canvas(img, target):
    """Upscale small photos so thin parts stay visible; big photos keep their own resolution (capped)."""
    H0, W0 = img.shape[:2]
    up = target / max(H0, W0)
    up = max(up, 1.0) if max(H0, W0) < target else min(1.0, 4000 / max(H0, W0))
    out = cv2.resize(img, None, fx=up, fy=up, interpolation=cv2.INTER_CUBIC if up > 1 else cv2.INTER_AREA) if up != 1 else img.copy()
    return out, up


def _style(H, W):
    """Line width and font size that grow with the picture, so they look the same on any photo."""
    side = max(H, W)
    return max(2, round(side / 900)), max(0.7, side / 2600)


def _legend(rows, total, col, H, title):
    ph = max(H, 54 * (len(rows) + 3))
    panel = np.full((ph, 420, 3), 28, np.uint8)
    cv2.putText(panel, title, (18, 44), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(panel, f"Total: {total}", (18, 92), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2, cv2.LINE_AA)
    for r, (k, n) in enumerate(rows):
        y = 150 + r * 48
        cv2.rectangle(panel, (18, y - 26), (52, y + 8), col[k], -1)
        cv2.putText(panel, f"{k}: {n}", (66, y + 2), cv2.FONT_HERSHEY_SIMPLEX, 0.95, (235, 235, 235), 2, cv2.LINE_AA)
    return panel


def _join(out, panel):
    H, W = out.shape[:2]
    ph = panel.shape[0]
    left = np.vstack([out, np.zeros((ph - H, W, 3), np.uint8)]) if ph > H else out
    return np.hstack([left, panel])


def draw_elements(img, boxes, target=2400, fill=0.28):
    """Step 1 picture: WHERE the parts are. Every part in one colour, thick outline and a light fill."""
    out, up = _canvas(img, target)
    H, W = out.shape[:2]
    lw, _ = _style(H, W)
    bx = (boxes * up).numpy() if len(boxes) else np.zeros((0, 4))
    over = out.copy()
    for x0, y0, x1, y1 in bx.round().astype(int):
        cv2.rectangle(over, (x0, y0), (x1, y1), ELEMENT_COLOUR, -1)
    out = cv2.addWeighted(over, fill, out, 1 - fill, 0)
    for x0, y0, x1, y1 in bx.round().astype(int):
        cv2.rectangle(out, (x0, y0), (x1, y1), (0, 0, 0), lw + 2, cv2.LINE_AA)       # dark rim: readable on any board colour
        cv2.rectangle(out, (x0, y0), (x1, y1), ELEMENT_COLOUR, lw, cv2.LINE_AA)
    return _join(out, _legend([("element", len(bx))], len(bx), {"element": ELEMENT_COLOUR}, H, "Elements"))


def draw(img, boxes, labels, scores, mode, min_label_px, target=2400):
    """Step 2 picture: WHAT each part is. Colour per type, thick boxes, class codes on boxes big enough to carry
    them (never overlapping), and a legend panel with the counts.
    mode: none | auto | ids | names"""
    out, up = _canvas(img, target)
    H, W = out.shape[:2]
    lw, fs = _style(H, W)
    col = palette(labels)
    bx = (boxes * up).numpy() if len(boxes) else np.zeros((0, 4))
    for i in range(len(bx)):
        x0, y0, x1, y1 = [int(round(v)) for v in bx[i]]
        cv2.rectangle(out, (x0, y0), (x1, y1), (0, 0, 0), lw + 2, cv2.LINE_AA)
        cv2.rectangle(out, (x0, y0), (x1, y1), col[labels[i]], lw, cv2.LINE_AA)
    if mode != "none":
        font, th_txt = cv2.FONT_HERSHEY_SIMPLEX, max(1, lw - 1)
        taken = []
        area = (bx[:, 2] - bx[:, 0]) * (bx[:, 3] - bx[:, 1]) if len(bx) else []
        for i in (np.argsort(-np.asarray(area)) if len(bx) else []):
            x0, y0, x1, y1 = [int(round(v)) for v in bx[i]]
            if mode == "auto" and min(x1 - x0, y1 - y0) < min_label_px * up:
                continue
            text = {"ids": str(i + 1), "names": f"{labels[i]} {scores[i]:.2f}"}.get(mode, SHORT.get(labels[i], labels[i][:3]))
            if not text:
                continue
            (tw, th), _ = cv2.getTextSize(text, font, fs, th_txt)
            for tx, ty in ((x0, y0 - 4), (x0, y1 + th + 4), (x1 - tw, y0 - 4), (x1 - tw, y1 + th + 4)):
                rect = (tx - 2, ty - th - 2, tx + tw + 2, ty + 4)
                if rect[0] < 0 or rect[1] < 0 or rect[2] > W or rect[3] > H:
                    continue
                if all(rect[2] < r[0] or rect[0] > r[2] or rect[3] < r[1] or rect[1] > r[3] for r in taken):
                    taken.append(rect)
                    cv2.putText(out, text, (tx, ty), font, fs, (0, 0, 0), th_txt + 3, cv2.LINE_AA)
                    cv2.putText(out, text, (tx, ty), font, fs, col[labels[i]], th_txt, cv2.LINE_AA)
                    break
    counts = {k: labels.count(k) for k in col}
    rows = sorted(counts.items(), key=lambda kv: -kv[1])
    return _join(out, _legend(rows, len(labels), col, H, "Types")), counts


# --------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--source", required=True, help="image or folder")
    ap.add_argument("--mode", choices=["any", "fine", "both"], default="both")
    ap.add_argument("--any", dest="any_w", help="weights of the 'component' detector")
    ap.add_argument("--fine", dest="fine_w", help="weights of the multi-class detector")
    ap.add_argument("--conf", type=float, default=0.20, help="lower = more elements (and more false ones)")
    ap.add_argument("--iou", type=float, default=0.5, help="NMS IoU when merging tiles / agnostic NMS")
    ap.add_argument("--imgsz", type=int, default=1024)
    ap.add_argument("--tile", type=int, default=0, help="tile size in px for big photos (0 = off), e.g. 640")
    ap.add_argument("--overlap", type=float, default=0.25)
    ap.add_argument("--max-det", type=int, default=1000, help="max detections per image / tile")
    ap.add_argument("--scale", type=float, default=1.0, help="resize the photo first (e.g. 0.5 when the parts look much "
                    "bigger than in the training tiles); boxes are reported in original pixels")
    ap.add_argument("--labels", choices=["none", "ids", "names", "auto"], default="auto")
    ap.add_argument("--min-label-px", type=int, default=24, help="a box needs this many px (at 640) on its short side to get a text label")
    ap.add_argument("--refine", help="python file with classify(list_of_bgr_crops)->list[(label, conf)] for the '?' boxes")
    ap.add_argument("--out", default="infer_out")
    ap.add_argument("--device", default=None)
    a = ap.parse_args()

    need_any = a.mode in ("any", "both"); need_fine = a.mode in ("fine", "both")
    if need_any and not a.any_w: ap.error("--any weights are required for this mode")
    if need_fine and not a.fine_w: ap.error("--fine weights are required for this mode")
    m_any = YOLO(a.any_w) if need_any else None
    m_fine = YOLO(a.fine_w) if need_fine else None
    fine_names = list(m_fine.names.values()) if m_fine else []

    refine = None
    if a.refine:
        import importlib.util
        spec = importlib.util.spec_from_file_location("refine_mod", a.refine)
        mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod); refine = mod.classify

    src = Path(a.source)
    files = sorted(p for p in src.iterdir() if p.suffix.lower() in IMG_EXT) if src.is_dir() else [src]
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)

    for f in files:
        img = cv2.imread(str(f))
        if img is None:
            print(f"{f.name}: cannot be read - skipped")
            continue
        orig = img
        if a.scale != 1.0:
            img = cv2.resize(img, None, fx=a.scale, fy=a.scale, interpolation=cv2.INTER_AREA if a.scale < 1 else cv2.INTER_LINEAR)
        if a.mode == "any":
            B, S, C = agnostic_nms(*detect(m_any, img, a.conf, a.imgsz, a.tile, a.overlap, a.iou, a.device, a.max_det), a.iou)
            labels, cls_conf = ["component"] * len(B), [float(s) for s in S]
        elif a.mode == "fine":
            B, S, C = detect(m_fine, img, a.conf, a.imgsz, a.tile, a.overlap, a.iou, a.device, a.max_det)
            labels, cls_conf = [fine_names[int(c)] for c in C], [float(s) for s in S]
        else:
            da = agnostic_nms(*detect(m_any, img, a.conf, a.imgsz, a.tile, a.overlap, a.iou, a.device, a.max_det), a.iou)
            df = detect(m_fine, img, a.conf, a.imgsz, a.tile, a.overlap, a.iou, a.device, a.max_det)
            B, S, _ = da
            labels, cls_conf = combine(da, df, fine_names)
        # back to the original photo: boxes in its pixels, crops for --refine at full resolution
        if a.scale != 1.0:
            B = B / a.scale
            img = orig
        if refine is not None:
            idx = [i for i, l in enumerate(labels) if l == "?"]
            if idx:
                crops = [img[max(int(B[i][1]), 0):int(B[i][3]), max(int(B[i][0]), 0):int(B[i][2])] for i in idx]
                for i, (lab, c) in zip(idx, refine(crops)):
                    labels[i], cls_conf[i] = lab, c

        stem = f.stem
        # 1) where the parts are (one colour)  2) what they are (colour per type)
        cv2.imwrite(str(out / f"{stem}_elements.jpg"), draw_elements(img, B), [cv2.IMWRITE_JPEG_QUALITY, 92])
        vis, counts = draw(img, B, labels, [float(s) for s in S], a.labels, a.min_label_px)
        if a.mode != "any":
            cv2.imwrite(str(out / f"{stem}_types.jpg"), vis, [cv2.IMWRITE_JPEG_QUALITY, 92])
        with open(out / f"{stem}_detections.csv", "w", newline="", encoding="utf8") as fh:
            w = csv.writer(fh); w.writerow(["id", "label", "box_conf", "label_conf", "x0", "y0", "x1", "y1"])
            for i in range(len(B)):
                w.writerow([i + 1, labels[i], f"{float(S[i]):.3f}", f"{cls_conf[i]:.3f}", *[int(v) for v in B[i]]])
        (out / f"{stem}_summary.json").write_text(json.dumps({"image": f.name, "total": len(B), "counts": counts}, indent=2))
        print(f"{f.name}: {len(B)} elements  {counts}")


if __name__ == "__main__":
    main()

## 3) داده
اگر دیتاست قبلاً باز شده باشد این بلوک چیزی را دوباره کپی نمی‌کند.

In [ ]:
import zipfile, shutil, glob
DATA = f"{WORK}/final_dataset_hr"
ZIP_NAME = "pcb_data_hr.zip"
if os.path.exists(f"{DATA}/fine/data.yaml"):
    print("دیتاست از قبل آماده است - رد شد")
else:
    local = f"{WORK}/{ZIP_NAME}"
    src = f"{DRIVE_DIR}/{ZIP_NAME}"
    if not os.path.exists(src):
        found = [f for pat in (f"/content/drive/MyDrive/{ZIP_NAME}", f"/content/drive/MyDrive/*/{ZIP_NAME}",
                               f"/content/drive/.shortcut-targets-by-id/*/*/{ZIP_NAME}") for f in glob.glob(pat)]
        src = found[0] if found else None
    if src:
        print(f"کپی {src} روی دیسک سریع Colab ...")
        shutil.copy2(src, local)
    elif ZIP_FILE_ID:
        print(f"{ZIP_NAME} در Drive نبود - دانلود از لینک ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "gdown"], check=True)
        import gdown
        gdown.download(id=ZIP_FILE_ID, output=local, quiet=False)
    else:
        raise FileNotFoundError(f"{ZIP_NAME} در Google Drive پیدا نشد - آن را در {DRIVE_DIR} بگذار")
    if not zipfile.is_zipfile(local):
        raise RuntimeError("فایل zip معتبر نیست - دسترسی لینک یا آپلود را چک کن")
    with zipfile.ZipFile(local) as z:
        bad = z.testzip()
        if bad:
            raise RuntimeError(f"فایل zip خراب است ({bad}) - دوباره آپلودش کن")
        print("باز کردن ...")
        z.extractall(WORK)
    os.remove(local)
    print("دیتاست باز شد:", DATA)
# the any/ view uses the same pictures as fine/: link them (no extra disk space) when the zip stores them once
for sp in ("train", "val", "test"):
    src_dir, dst_dir = f"{DATA}/fine/images/{sp}", f"{DATA}/any/images/{sp}"
    if os.path.isdir(src_dir) and not os.path.isdir(dst_dir):
        os.makedirs(dst_dir)
        for f in os.listdir(src_dir):
            try:
                os.link(f"{src_dir}/{f}", f"{dst_dir}/{f}")
            except OSError:
                shutil.copy2(f"{src_dir}/{f}", f"{dst_dir}/{f}")
        print(f"any/images/{sp}: {len(os.listdir(dst_dir))} عکس")
if os.path.exists(f"{DATA}/report.txt"):
    print(open(f"{DATA}/report.txt", encoding="utf8").read()[:2500])

## 4) بررسی داده (قبل از مصرف وقت GPU)
برچسب‌ها، رنگی بودن همه‌ی عکس‌ها، و اینکه یک برد در دو بخش train/val/test نباشد. اگر خطا باشد اینجا متوقف می‌شود.

In [ ]:
rc = run([sys.executable, f"{WORK}/verify_dataset.py", "--root", DATA], check=False)
assert rc == 0, "دیتاست مشکل دارد - گزارش بالا را بخوان و آموزش را شروع نکن"

## 5) وضعیت آموزش
بعد از قطع شدن، قبل از ادامه ببین کار تا کجا رفته (چیزی را اجرا یا تغییر نمی‌دهد).

In [ ]:
import csv, json, yaml
from IPython.display import Image, display
for stage in STAGES.split():
    d = f"{RUNS}/{stage}_{os.path.splitext(MODEL)[0]}"
    rows = [{k.strip(): v for k, v in r.items()} for r in csv.DictReader(open(f"{d}/results.csv"))] if os.path.exists(f"{d}/results.csv") else []
    rows = list({int(float(r["epoch"])): r for r in rows}.values())   # repeated epochs after a fallback: keep the latest
    total = yaml.safe_load(open(f"{d}/args.yaml"))["epochs"] if os.path.exists(f"{d}/args.yaml") else EPOCHS
    if os.path.exists(f"{d}/DONE"):
        state = "تمام شده"
        if os.path.exists(f"{d}/test_metrics.json"):
            m = json.load(open(f"{d}/test_metrics.json", encoding="utf8"))
            state += f" | TEST mAP50 {m['mAP50']:.3f}  mAP50-95 {m['mAP50-95']:.3f}"
    elif rows:
        best = max(rows, key=lambda r: float(r["metrics/mAP50-95(B)"]))
        state = (f"در حال آموزش: epoch {len(rows)} از {total} | بهترین mAP50-95 تا الان {float(best['metrics/mAP50-95(B)']):.3f} "
                 f"(epoch {best['epoch'].strip()}) | mAP50 {float(best['metrics/mAP50(B)']):.3f}")
    else:
        state = "هنوز شروع نشده"
    print(f"[{stage}] {state}")
    if os.path.exists(f"{d}/progress.png"):
        display(Image(f"{d}/progress.png", width=1100))

## 6) آموزش
مرحله‌ی `any`: همه‌ی المان‌ها را بدون توجه به نوع پیدا می‌کند. مرحله‌ی `fine`: نوع را مشخص می‌کند (از وزن‌های `any` شروع می‌شود).
هر epoch روی Drive ذخیره می‌شود. قطع شد؟ همین بلوک را دوباره اجرا کن.

خروجی زنده است: بعد از هر epoch یک خط خلاصه می‌آید (loss، دقت روی val، بهترین epoch، زمان هر epoch و زمان باقی‌مانده). نمودار زنده هم بالای خروجی می‌آید و بعد از هر epoch به‌روز می‌شود. همین‌ها روی Drive هم هست تا از گوشی ببینی: `<RUNS>/<stage>_<model>/progress.txt` و `progress.png`.

**ایمنی:** قبل از ذخیره‌ی هر epoch، نسخه‌ی قبلی `last.pt` در `last_prev.pt` نگه داشته می‌شود؛ اگر قطع شدن وسط ذخیره فایل را خراب کند، خودش از نسخه‌ی قبلی ادامه می‌دهد.

In [ ]:
# --- ایمنی: ادامه از آموزش قبلی، نه شروع از صفر ---
# (مسیر داخل checkpoint را خود اسکریپت آموزش با این پوشه یکی می‌کند، پس آموزش حساب/پوشه‌ی دیگر هم درست ادامه پیدا می‌کند)
import glob as _g, torch as _t
_ck = _g.glob(f"{RUNS}/*/weights/last.pt")
print("RUNS =", RUNS, flush=True)
print("محتوای پوشه:", sorted(os.listdir(RUNS)), flush=True)
for stage in STAGES.split():
    _d = f"{RUNS}/{stage}_{os.path.splitext(MODEL)[0]}"
    _w = sorted(os.listdir(f"{_d}/weights")) if os.path.isdir(f"{_d}/weights") else []
    if os.path.exists(f"{_d}/DONE"):
        _s = "تمام شده (DONE) -> رد می‌شود"
    elif "last.pt" in _w:
        try:
            _e = _t.load(f"{_d}/weights/last.pt", map_location="cpu", weights_only=False).get("epoch", -1)
            _s = f"ادامه از epoch {_e + 2}" if _e >= 0 else "آموزش تمام شده، فقط تست می‌شود"
        except Exception as ex:
            _s = f"last.pt خراب است ({ex}) -> از نسخه‌ی پشتیبان ادامه می‌دهد"
    else:
        _s = "last.pt ندارد -> از اول شروع می‌شود"
    print(f"[{stage}] {_d} | weights: {_w} | => {_s}", flush=True)
if not _ck and not os.path.exists(f"{RUNS}/final/fine_best.pt"):
    assert ALLOW_NEW_RUN, ("❌ هیچ آموزش قبلی (last.pt) در پوشه‌ی پروژه نیست؛ آموزش از صفر شروع می‌شد. "
                           "اگر آموزش قبلی جای دیگری است مسیرش را در RUNS_DIR بنویس. اگر واقعا آموزش جدید می‌خواهی ALLOW_NEW_RUN = True")
cmd = [sys.executable, f"{WORK}/train_yolo26.py", "--data-root", DATA, "--project", RUNS,
       "--stages", *STAGES.split(), "--model", MODEL, "--imgsz", str(IMGSZ), "--epochs", str(EPOCHS), *EXTRA.split()]
if SESSION_HOURS:
    cmd += ["--hours", str(SESSION_HOURS)]
if BATCH:
    cmd += ["--batch", str(BATCH)]
if not GPU:
    print("⚠️ بدون GPU: آموزش روی CPU (هر epoch چند ساعت). وزن‌ها فقط آخر هر epoch ذخیره می‌شوند.")
    cmd += ["--device", "cpu", "--allow-cpu", "--workers", "2"]
run(cmd, watch=f"{RUNS}/*_{os.path.splitext(MODEL)[0]}/progress.png")

## 7) نتیجه

In [ ]:
import json, glob
for stage in STAGES.split():
    p = f"{RUNS}/{stage}_{os.path.splitext(MODEL)[0]}/test_metrics.json"
    if os.path.exists(p):
        m = json.load(open(p, encoding="utf8"))
        print(f"[{stage}] TEST  P {m['precision']:.3f}  R {m['recall']:.3f}  mAP50 {m['mAP50']:.3f}  mAP50-95 {m['mAP50-95']:.3f}")
        pc = m.get("per_class", {})
        for n, v in sorted(pc.items(), key=lambda kv: -kv[1]["mAP50"]) if len(pc) > 1 else []:
            print(f"     {n:<24} P {v['precision']:.3f}  R {v['recall']:.3f}  mAP50 {v['mAP50']:.3f}")
print("بهترین وزن‌ها:", glob.glob(f"{RUNS}/final/*.pt"))

out = f"{RUNS}/test_preview"
if not glob.glob(out + "/*_elements.jpg") and os.path.exists(f"{RUNS}/final/any_best.pt"):
    mode = "both" if os.path.exists(f"{RUNS}/final/fine_best.pt") else "any"
    args = ["--any", f"{RUNS}/final/any_best.pt", "--mode", mode, "--source", f"{DATA}/boards_test/images", "--out", out,
            "--tile", "1024", "--imgsz", str(IMGSZ)]
    if mode == "both":
        args += ["--fine", f"{RUNS}/final/fine_best.pt"]
    run([sys.executable, f"{WORK}/predict.py", *args])
try:
    from IPython.display import Image, display
    for p in sorted(glob.glob(out + "/*_elements.jpg"))[:3]:
        print("1) جای المان‌ها:", os.path.basename(p)); display(Image(p, width=1400))
        t = p.replace("_elements.jpg", "_types.jpg")
        if os.path.exists(t):
            print("2) نوع المان‌ها:"); display(Image(t, width=1400))
except ImportError:
    pass

## 8) تست روی عکس خودت
`IMAGE` را خالی بگذار تا دکمه‌ی آپلود بیاید، یا مسیر عکس/پوشه را بنویس. `MODE`: `any` (هر المان)، `fine` (فقط نام‌دار)، `both` (پیدا کن و نام ببر؛ ناشناخته‌ها `?`).

In [ ]:
IMAGE = ""        # مثلا "/content/drive/MyDrive/pcb/my_board.jpg" یا خالی برای آپلود
MODE  = "both"
CONF  = 0.20      # کمتر = المان بیشتر (و اشتباه بیشتر)
TILE  = 1024      # 1024 = عکس به تکه‌های 1024 پیکسلی (مثل آموزش) بریده می‌شود؛ 0 = کل عکس یکجا
SCALE = 1.0       # اگر قطعه‌ها در عکس خیلی بزرگ‌اند (عکس خیلی نزدیک) 0.5 بگذار

any_w, fine_w = f"{RUNS}/final/any_best.pt", f"{RUNS}/final/fine_best.pt"
if not IMAGE:
    try:
        from google.colab import files
        up = files.upload(); IMAGE = os.path.join(WORK, list(up)[0]); open(IMAGE, "wb").write(list(up.values())[0])
    except ImportError:
        print("آدرس عکس را در IMAGE بنویس")
if IMAGE and os.path.exists(any_w):
    out = f"{WORK}/my_test"
    args = ["--source", IMAGE, "--mode", MODE, "--conf", str(CONF), "--tile", str(TILE), "--scale", str(SCALE),
            "--imgsz", str(IMGSZ), "--out", out, "--any", any_w]
    if MODE in ("fine", "both"):
        args += ["--fine", fine_w]
    run([sys.executable, f"{WORK}/predict.py", *args])
    try:
        from IPython.display import Image, display
        for p in sorted(glob.glob(out + "/*_elements.jpg")):
            print("1) جای المان‌ها (یک رنگ):", os.path.basename(p)); display(Image(p, width=1400))
            t = p.replace("_elements.jpg", "_types.jpg")
            if os.path.exists(t):
                print("2) نوع هر المان:"); display(Image(t, width=1400))
    except ImportError:
        pass